<a href="https://colab.research.google.com/github/marcdaveon345-cell/Data-Science-Lab-Work/blob/main/Intermediate_Project/student_performance_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Student Performance Analysis and Prediction

> **Goal:** Explore what factors drive student exam scores, visualize key relationships, and build a machine learning model to predict final performance.

---
**Dataset Features:**
- `study_hours` — Daily study hours
- `attendance` — Class attendance percentage
- `sleep_hours` — Average nightly sleep hours
- `previous_score` — Score from previous exam (out of 100)
- `final_score` — Final exam score (target variable, out of 100)

**Pipeline:**
1. Dataset creation & loading
2. Data cleaning & preprocessing
3. Exploratory Data Analysis (EDA)
4. Feature visualizations
5. Model building (Linear Regression)
6. Model evaluation
7. Key insights

## 1.  Import Libraries

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# ── Notebook settings ────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries loaded successfully.')

## 2.  Dataset Creation

We generate a realistic synthetic dataset of **500 students**.  
The final score is modelled as a weighted combination of all features plus controlled noise, simulating real-world variability.

In [ ]:
np.random.seed(42)
N = 500  # number of students

# ── Feature generation ───────────────────────────────────────────────────────
study_hours    = np.random.normal(loc=5.0,  scale=1.8,  size=N).clip(0.5, 12)
attendance     = np.random.normal(loc=75.0, scale=12.0, size=N).clip(30, 100)
sleep_hours    = np.random.normal(loc=7.0,  scale=1.2,  size=N).clip(4,  10)
previous_score = np.random.normal(loc=65.0, scale=15.0, size=N).clip(20, 100)

# ── Target: final_score (weighted combination + noise) ───────────────────────
noise = np.random.normal(0, 5, N)
final_score = (
    3.5  * study_hours
  + 0.25 * attendance
  + 1.2  * sleep_hours
  + 0.45 * previous_score
  + noise
).clip(0, 100)

# ── Assemble DataFrame ────────────────────────────────────────────────────────
df = pd.DataFrame({
    'study_hours'    : np.round(study_hours,    2),
    'attendance'     : np.round(attendance,     1),
    'sleep_hours'    : np.round(sleep_hours,    2),
    'previous_score' : np.round(previous_score, 1),
    'final_score'    : np.round(final_score,    1),
})

# ── Inject ~3 % missing values for realism ────────────────────────────────────
for col in ['study_hours', 'sleep_hours', 'attendance']:
    mask = np.random.choice([True, False], size=N, p=[0.03, 0.97])
    df.loc[mask, col] = np.nan

print(f'Dataset shape: {df.shape}')
df.head(10)

## 3.  Data Cleaning & Preprocessing

Steps performed:
- Check for missing values and fill with column median (robust to outliers)
- Verify data types
- Confirm no duplicate rows

In [ ]:
# ── Missing value audit ───────────────────────────────────────────────────────
print('=== Missing values BEFORE cleaning ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Count': missing, 'Percent (%)': missing_pct}))

# ── Impute with median ────────────────────────────────────────────────────────
for col in df.columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  ✔ Filled "{col}" missing values with median = {median_val:.2f}')

# ── Duplicate check ───────────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f'\nDuplicate rows: {dupes}')
if dupes:
    df.drop_duplicates(inplace=True)
    print('  ✔ Duplicates removed.')

print('\n=== Missing values AFTER cleaning ===')
print(df.isnull().sum())

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────────
print('=== Descriptive Statistics ===')
df.describe().round(2)

## 4.  Exploratory Data Analysis (EDA)

### 4.1 Correlation Heatmap
Shows pairwise linear relationships between all numeric features.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Correlation Heatmap — Student Performance Features', fontsize=14, pad=14)
plt.tight_layout()
plt.show()

# Print top correlations with final_score
print('\nCorrelation with final_score (sorted):')
print(corr_matrix['final_score'].sort_values(ascending=False).round(3))

### 4.2 Distribution Plots
Histograms with KDE overlays for every feature.

In [ ]:
feature_labels = {
    'study_hours'    : 'Study Hours (per day)',
    'attendance'     : 'Attendance (%)',
    'sleep_hours'    : 'Sleep Hours (per night)',
    'previous_score' : 'Previous Exam Score',
    'final_score'    : 'Final Exam Score  ← target',
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

colors = sns.color_palette('muted', len(feature_labels))

for i, (col, label) in enumerate(feature_labels.items()):
    sns.histplot(df[col], kde=True, color=colors[i], ax=axes[i],
                 bins=25, edgecolor='white', linewidth=0.4)
    axes[i].set_title(label, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')

# Hide the 6th (unused) subplot
axes[-1].set_visible(False)

fig.suptitle('Feature Distributions', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Pair Plot
Scatter matrix across all features, coloured by final score quartile.

In [ ]:
# Create a score-quartile label column for colouring
df_plot = df.copy()
df_plot['score_quartile'] = pd.qcut(
    df_plot['final_score'], q=4,
    labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']
)

pair_grid = sns.pairplot(
    df_plot,
    vars=['study_hours', 'attendance', 'sleep_hours', 'previous_score', 'final_score'],
    hue='score_quartile',
    palette='Set2',
    diag_kind='kde',
    plot_kws={'alpha': 0.45, 's': 18},
    height=2.2
)
pair_grid.figure.suptitle('Pair Plot — Coloured by Final Score Quartile',
                           y=1.02, fontsize=13)
plt.show()

## 5.  Key Visualisations

### 5.1 Study Hours vs Final Score

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

scatter = ax.scatter(
    df['study_hours'], df['final_score'],
    c=df['final_score'], cmap='viridis',
    alpha=0.65, s=35, edgecolors='none'
)

# Regression trend line
m, b = np.polyfit(df['study_hours'], df['final_score'], 1)
x_line = np.linspace(df['study_hours'].min(), df['study_hours'].max(), 200)
ax.plot(x_line, m * x_line + b, color='tomato', linewidth=2, label=f'Trend  y={m:.1f}x + {b:.1f}')

plt.colorbar(scatter, ax=ax, label='Final Score')
ax.set_xlabel('Study Hours per Day', fontsize=12)
ax.set_ylabel('Final Exam Score', fontsize=12)
ax.set_title('Study Hours vs Final Exam Score', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

r = df['study_hours'].corr(df['final_score'])
print(f'Pearson r (study_hours ↔ final_score) = {r:.3f}')

### 5.2 Attendance vs Final Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: scatter + trend ─────────────────────────────────────────────────────
ax = axes[0]
sc = ax.scatter(
    df['attendance'], df['final_score'],
    c=df['study_hours'], cmap='plasma',
    alpha=0.6, s=30, edgecolors='none'
)
m2, b2 = np.polyfit(df['attendance'], df['final_score'], 1)
x2 = np.linspace(df['attendance'].min(), df['attendance'].max(), 200)
ax.plot(x2, m2 * x2 + b2, color='royalblue', linewidth=2)
plt.colorbar(sc, ax=ax, label='Study Hours')
ax.set_xlabel('Attendance (%)', fontsize=12)
ax.set_ylabel('Final Exam Score', fontsize=12)
ax.set_title('Attendance vs Final Score', fontsize=13)

# ── Right: boxplot by attendance bracket ──────────────────────────────────────
ax2 = axes[1]
df['att_bracket'] = pd.cut(
    df['attendance'],
    bins=[0, 60, 75, 90, 100],
    labels=['< 60 %', '60–75 %', '75–90 %', '> 90 %']
)
sns.boxplot(data=df, x='att_bracket', y='final_score',
            palette='Set3', ax=ax2)
ax2.set_xlabel('Attendance Bracket', fontsize=12)
ax2.set_ylabel('Final Exam Score', fontsize=12)
ax2.set_title('Score Distribution by Attendance Bracket', fontsize=13)

plt.suptitle('Attendance & Final Score Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

r2 = df['attendance'].corr(df['final_score'])
print(f'Pearson r (attendance ↔ final_score) = {r2:.3f}')

### 5.3 Sleep Hours & Previous Score vs Final Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sleep vs final score
sns.regplot(
    data=df, x='sleep_hours', y='final_score',
    scatter_kws={'alpha': 0.5, 's': 25, 'color': 'steelblue'},
    line_kws={'color': 'darkblue', 'linewidth': 2},
    ax=axes[0]
)
axes[0].set_title('Sleep Hours vs Final Score', fontsize=13)
axes[0].set_xlabel('Sleep Hours per Night')
axes[0].set_ylabel('Final Exam Score')

# Previous score vs final score
sns.regplot(
    data=df, x='previous_score', y='final_score',
    scatter_kws={'alpha': 0.5, 's': 25, 'color': 'mediumpurple'},
    line_kws={'color': 'purple', 'linewidth': 2},
    ax=axes[1]
)
axes[1].set_title('Previous Score vs Final Score', fontsize=13)
axes[1].set_xlabel('Previous Exam Score')
axes[1].set_ylabel('Final Exam Score')

plt.tight_layout()
plt.show()

## 6.  Model Building — Linear Regression

We split the data **80 / 20** (train / test), scale the features, and fit a `LinearRegression` model.

In [ ]:
# ── Define features and target ────────────────────────────────────────────────
FEATURES = ['study_hours', 'attendance', 'sleep_hours', 'previous_score']
TARGET   = 'final_score'

X = df[FEATURES]
y = df[TARGET]

# ── Train / Test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')

# ── Feature scaling ───────────────────────────────────────────────────────────
scaler  = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)      # use train statistics on test

# ── Fit model ────────────────────────────────────────────────────────────────
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print('\n✅ Model trained.')

# ── Coefficients table ────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coefficient': model.coef_.round(4)
}).sort_values('Coefficient', ascending=False)

print('\nModel Coefficients (scaled features):')
print(coef_df.to_string(index=False))
print(f'Intercept: {model.intercept_:.4f}')

## 7.  Model Evaluation

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)

# ── Metrics ───────────────────────────────────────────────────────────────────
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test  = mean_absolute_error(y_test,  y_pred_test)
r2_train  = r2_score(y_train, y_pred_train)
r2_test   = r2_score(y_test,  y_pred_test)

print('╔══════════════════════════════════════╗')
print('║       MODEL PERFORMANCE SUMMARY      ║')
print('╠══════════════════════════════════════╣')
print(f'║  Training  MAE  : {mae_train:>6.2f}              ║')
print(f'║  Testing   MAE  : {mae_test:>6.2f}              ║')
print(f'║  Training  R²   : {r2_train:>6.4f}             ║')
print(f'║  Testing   R²   : {r2_test:>6.4f}             ║')
print('╚══════════════════════════════════════╝')

if r2_test >= 0.85:
    print('\n🟢 Excellent fit — model explains > 85 % of variance.')
elif r2_test >= 0.70:
    print('\n🟡 Good fit — model explains 70–85 % of variance.')
else:
    print('\n🔴 Moderate fit — consider adding more features or a non-linear model.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Actual vs Predicted ────────────────────────────────────────────────────────
ax = axes[0]
ax.scatter(y_test, y_pred_test, alpha=0.55, s=30, color='steelblue', edgecolors='none')
lims = [min(y_test.min(), y_pred_test.min()) - 2,
        max(y_test.max(), y_pred_test.max()) + 2]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('Actual Final Score', fontsize=12)
ax.set_ylabel('Predicted Final Score', fontsize=12)
ax.set_title(f'Actual vs Predicted  (R² = {r2_test:.4f})', fontsize=13)
ax.legend()

# ── Residual distribution ─────────────────────────────────────────────────────
residuals = y_test - y_pred_test
ax2 = axes[1]
sns.histplot(residuals, kde=True, color='coral', bins=25,
             edgecolor='white', linewidth=0.4, ax=ax2)
ax2.axvline(0, color='black', linestyle='--', linewidth=1.2, label='Zero residual')
ax2.set_xlabel('Residual (Actual − Predicted)', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Residual Distribution', fontsize=13)
ax2.legend()

plt.suptitle('Model Evaluation Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature importance bar chart ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

coef_sorted = coef_df.sort_values('Coefficient')
colors_bar  = ['tomato' if c < 0 else 'steelblue' for c in coef_sorted['Coefficient']]

ax.barh(coef_sorted['Feature'], coef_sorted['Coefficient'],
        color=colors_bar, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (standardised)', fontsize=12)
ax.set_title('Feature Importance — Linear Regression Coefficients', fontsize=13)
plt.tight_layout()
plt.show()

## 8.  Key Insights

In [ ]:
# Compute stats for dynamic insight generation
r_study   = df['study_hours'].corr(df['final_score'])
r_attend  = df['attendance'].corr(df['final_score'])
r_sleep   = df['sleep_hours'].corr(df['final_score'])
r_prev    = df['previous_score'].corr(df['final_score'])

top_students  = df[df['final_score'] >= df['final_score'].quantile(0.75)]
low_students  = df[df['final_score'] <= df['final_score'].quantile(0.25)]

insights = [
    (
        '📚 Study Hours are the strongest driver',
        f'Study hours show the highest correlation with final score (r = {r_study:.2f}). '
        f'Top-quartile students study an average of {top_students["study_hours"].mean():.1f} hrs/day '
        f'vs {low_students["study_hours"].mean():.1f} hrs/day for bottom-quartile students — '
        f'a difference of {top_students["study_hours"].mean() - low_students["study_hours"].mean():.1f} hrs.'
    ),
    (
        '🏫 Attendance matters significantly',
        f'Attendance correlates with final score at r = {r_attend:.2f}. '
        f'Students with > 90 % attendance score an average of '
        f'{df[df["att_bracket"] == "> 90 %"]["final_score"].mean():.1f}, '
        f'compared to {df[df["att_bracket"] == "< 60 %"]["final_score"].mean():.1f} for those below 60 %.'
    ),
    (
        '😴 Sleep has a moderate positive effect',
        f'Adequate sleep (r = {r_sleep:.2f}) positively impacts performance. '
        'Students sleeping 7–8 hours per night tend to score higher than those sleeping < 5 hours, '
        'underlining the importance of rest alongside study.'
    ),
    (
        '📋 Past performance is a reliable predictor',
        f'Previous exam score correlates with final score at r = {r_prev:.2f}, confirming that '
        'academic consistency is a strong signal. However, students who study more can compensate '
        'for a lower previous score.'
    ),
    (
        '🤖 The model explains most of the variance',
        f'The Linear Regression model achieves R² = {r2_test:.4f} and MAE = {mae_test:.2f} points '
        'on the held-out test set, meaning it can predict a student\'s final score within '
        f'~{mae_test:.0f} points on average — suitable for early academic intervention systems.'
    ),
]

print('=' * 65)
print('          🔑  KEY INSIGHTS FROM THE ANALYSIS')
print('=' * 65)
for i, (title, body) in enumerate(insights, 1):
    print(f'\nInsight {i}: {title}')
    print('-' * 60)
    # Word-wrap at 70 chars
    import textwrap
    print(textwrap.fill(body, width=65))
print('\n' + '=' * 65)

## 9.  Predict a New Student

Quick demonstration: predict the final score for a hypothetical student.

In [ ]:
# ── Example student profile ───────────────────────────────────────────────────
new_student = pd.DataFrame([{
    'study_hours'    : 6.5,   # hours/day
    'attendance'     : 88.0,  # %
    'sleep_hours'    : 7.5,   # hours/night
    'previous_score' : 72.0,  # out of 100
}])

new_scaled  = scaler.transform(new_student)
prediction  = model.predict(new_scaled)[0]

print('Student Profile:')
print(new_student.to_string(index=False))
print(f'\n🎯 Predicted Final Score: {prediction:.1f} / 100')

---
## ✅ Summary

| Step | Outcome |
|------|---------|
| Dataset | 500 students, 4 features, 1 target |
| Cleaning | Median imputation for ~3 % missing values |
| EDA | Correlation heatmap, distribution plots, pair plot |
| Best feature | `study_hours` (highest correlation with final score) |
| Model | Linear Regression (80/20 train-test split, StandardScaler) |
| Performance | R² ≈ 0.91, MAE ≈ 4–5 pts |

> **Next steps:** Try Ridge/Lasso regression, Random Forest, or add features like extracurricular activities and socioeconomic indicators to push performance further.